# Lab 12: Getting Tables Out of Prose

**DSA 405 · Week 12**

| | |
|---|---|
| **In class** | Friday, Nov 6 |
| **Take-home** | none (**P4 work session**) |
| **Files** | `restaurant_notes.txt`, three `inspection_report_*.pdf` |
| **P4 due** | Friday, Nov 20, 11:59 PM |
| **Time** | ~40 min in class, then the rest of the week on P4 |

> **No A this week.** Submit this notebook for in-class credit and spend the rest of the week
> on **P4**. Lightning talks are Nov 13 or Nov 20.

## Overview

This lab works with 24 inspection reports written as **prose**: ordinary sentences, not
a table. Each report contains nine structured fields, such as the score and the number
of critical violations. **Extraction** means finding each of those values inside the
sentences and putting it into a table column. Getting the values out takes little time,
whether you use a careful regex (a text-search pattern) or a language model (an AI tool
such as ChatGPT or Claude that reads text and writes text back). The harder part, and the
real subject of this lab, is writing a **schema**: a set of rules that says what values
each field is allowed to contain, so that a wrong extracted value gets caught even when
it looks reasonable. This is the verification habit from Week 7, applied to a messier
kind of input.

Every document was generated from a real row of `inspections_a.csv`, so your extraction
can be joined back to the source and scored field by field. Real extraction work never
comes with an answer key. This lab includes one so that you can compare your output
against the truth and learn what extraction errors look like.

In [ ]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, time, re, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users

HEADERS = {"User-Agent": "DSA405-student-coursework (NC State; teaching exercise)"}


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

---
## Task 1: Read the text before writing any code

The file contains twenty-four reports, separated by a header line. Read two of them in
full before writing any code. Every pattern you write will need an **anchor**: a fixed
word or phrase that sits next to the value in the text, such as "holds" just before the
seat count. The pattern searches for the anchor, so it matches the right number and not
some other number in the same report. The anchors are part of the sentence wording, and
the wording has small differences from one report to the next that you only notice by
reading.

In [ ]:
notes = load("restaurant_notes.txt", "text")

print("characters:", len(notes))
print("=" * 70)
print(notes[:900])

Before writing a pattern, list the nine visible fields and, for each, the surrounding
words to anchor on.

| Field | Anchor text |
|---|---|
| establishment | in the header line, before the comma |
| city | |
| inspection_date | |
| seats | |
| year_opened | |
| score | |
| critical_violations | |
| noncritical_violations | |
| reinspection_required | |

Anchoring on the surrounding words is what keeps a regex working on real text, where the
same kinds of numbers appear in many places.

## Task 2: Split into records, then extract one

Split the text into separate records, using the header line as the separator. Get the
extraction working completely on **one** record. Only then write the loop over all of
them. Most extraction bugs are easier to find on one record than on twenty-four.

In [ ]:
blocks = [b for b in notes.split("Inspection notes \u2014 ") if b.strip()]
print("records found:", len(blocks), " (should be 24)")
assert len(blocks) == 24, "record split is wrong; check the separator"

print()
print(blocks[0])

In [ ]:
b = blocks[0]

# header line: "Name, City"
head = b.split("\n")[0]
name, city = head.rsplit(",", 1)
print("establishment:", name.strip())
print("city         :", city.strip())

# now the numbers. Anchor each on the words around it.
print("seats        :", re.search(r"holds (\d+) seats", b).group(1))
print("year_opened  :", re.search(r"since (\d{4})", b).group(1))

### The period problem

Try to extract the score with the obvious pattern, and look at the result:

```python
re.search(r"score came to ([\d.]+)", b).group(1)
```

`[\d.]+` is **greedy**, which means it matches as many characters as it can. The period
is one of its allowed characters, so the match also captures the period that ends the
sentence. The result is `'86.7.'`, which is not a number, and `float()` raises an error
on it.

Fix it by describing the shape of the number instead of listing allowed characters:
`(\d+\.\d+)` means digits, then one dot, then digits. This is a general rule worth
remembering: describe the shape of the value you want, rather than listing the
characters it might contain, because a list of characters will eventually match
something you did not intend.

In [ ]:
greedy = re.search(r"score came to ([\d.]+)", b).group(1)
exact  = re.search(r"score came to (\d+\.\d+)", b).group(1)

print("greedy pattern gives:", repr(greedy))
print("anchored pattern    :", repr(exact))
try:
    float(greedy)
except ValueError as e:
    print("float(greedy) ->", type(e).__name__, e)
print("float(exact)  ->", float(exact))

## Task 3: Extract all 24, then verify against the source

Fill in the seven `...` lines inside `extract`. Each one needs a pattern with an anchor
from your Task 1 table. Be careful with singular and plural forms: the text contains
both "1 critical violation" and "3 critical violations", and a pattern that only matches
one of the two forms returns `None` for the other with no error message.

In [ ]:
def extract(block):
    """Pull nine structured fields out of one prose inspection report."""
    head = block.split("\n")[0]
    name, city = head.rsplit(",", 1)

    def grab(pattern):
        m = re.search(pattern, block)
        return m.group(1) if m else None

    return {
        "establishment": name.strip(),
        "city": city.strip(),
        "inspection_date": ...,
        "seats": ...,
        "year_opened": ...,
        "score": ...,
        "critical_violations": ...,
        "noncritical_violations": ...,
        "reinspection_required": ...,
    }


extracted = pd.DataFrame([extract(b) for b in blocks])
print(extracted.shape)
extracted.head()

Join the extraction back to `inspections_a.csv` and count, field by field, how many of
the 24 values match exactly.

In [ ]:
insp = load("inspections_a.csv")
b_clean = load("inspections_b.csv").drop_duplicates("restaurant_id")
source = insp.merge(b_clean, on="restaurant_id", how="left")

# The obvious join. Look carefully at the row count it returns.
check = extracted.merge(source, left_on="establishment",
                        right_on="restaurant_name", how="left")

print("extractions :", len(extracted))
print("joined rows :", len(check))

### The join returned the wrong number of rows. Find out why.

The extraction produced 24 records, and the join returned far more than 24. We saw this
pattern in Week 6.

**Three questions:**

1. Why did the row count grow? What is the grain of `source` (what one row represents),
   and what is the grain of `extracted`?
2. `establishment` is not a unique key in `source`. What is? The extraction contains a
   second field that, combined with the name, identifies exactly one inspection. The two
   tables write that field in different formats, so convert one of them before joining.
3. Redo the join on that composite key, and add `validate="one_to_one"` so that pandas
   raises an error if the key is still wrong.

What happened here is a **fan-out**: a join that creates extra copies of rows, the
silent error we studied in Week 6. This time the fan-out appeared inside the
verification step. A fan-out in checking code is worse than one in the analysis itself,
because it breaks the check that was supposed to catch errors, and then nothing is
checking your work at all.

## Task 4: A schema that catches bad output

Extraction produces *some* value for every field, even when the value is wrong. Now
write a **validator**: a function that applies your schema rules to the table and
reports every value that violates one. Every rule needs a comment stating what mistake
it protects against.

In [ ]:
def validate(df):
    """Check extracted records against what an inspection report can possibly say."""
    problems = []

    # protects against: a score that captured the sentence period, or a decimal-point error
    bad = df[~df.score.between(0, 100)]
    if len(bad):
        problems.append(f"{len(bad)} scores outside 0-100")

    # protects against: capturing the wrong number for the year
    # protects against: an establishment opening after it was inspected
    # protects against: negative or impossibly large violation counts
    # protects against: a missing field becoming None with no error message
    # ... at least three more

    return problems


print(validate(extracted) or "all records valid")

Show that the validator works: make a copy of the table, change three values in it in
three different wrong ways, and show the validator reporting each one. Each corruption
should be caught by a different rule.

In [ ]:
# corrupt the data three ways; each corruption should be caught by a different rule

## Task 5: The PDFs

The three PDFs contain the same nine fields, but in a file format that is harder to
read. `pypdf` extracts the text from a PDF, and after that this is the same problem you
just solved.

In [ ]:
!pip install -q pypdf

In [ ]:
from pypdf import PdfReader

FILES = ["inspection_report_00119.pdf",
         "inspection_report_00325.pdf",
         "inspection_report_00352.pdf"]

texts = {}
for f in FILES:
    raw = requests.get(DATA + f, timeout=30).content if DATA.startswith("http") else open(DATA + f, "rb").read()
    texts[f] = PdfReader(io.BytesIO(raw)).pages[0].extract_text()
    print(f"{f}: {len(texts[f])} characters")

In [ ]:
print(texts["inspection_report_00119.pdf"])

Extract the score from each PDF with the obvious approach:

```python
re.search(r"score of (\d+\.\d+)", text).group(1)
```

Then look closely at the third document.

In [ ]:
for f, t in texts.items():
    found = re.findall(r"score of (\d+\.\d+)", t)
    print(f"{f}: found {len(found)} score(s) -> {found}")

### The document with two scores

One of the documents contains two scores. It mentions an earlier inspection for context
before reporting the current one, and the earlier score appears **first** in the text.
So `re.search(...)` and `findall(...)[0]` both return the wrong value, and no error
message appears.

Read the document and decide which score is the correct one.

In [ ]:
print(texts["inspection_report_00352.pdf"])

**Three questions:**

1. Which score is correct, and what in the text tells you so?
2. What rule catches this problem? Name a check the validator can apply automatically
   to a document it has never seen, rather than advice like "read more carefully."
3. When the rule flags a document, what should happen next: guess, skip the document,
   or send it to a human to read? Explain your choice.

The idea behind question 2: when a pattern finds more matches than the schema allows,
that is useful information. A field that should contain exactly one value, but matches
two, is evidence that this document is different from the ones you designed your
patterns for.

*Your answers.*

In [ ]:
# implement the rule: flag any document where a field that should have one value matches more than one

## Task 6: Extract with a model, then check the model

If an AI assistant is available, paste one report into it and ask for the nine fields as
JSON. Then paste in the text of one of the PDFs, including the one with two scores.

Compare the model's output against the regex output and against `inspections_a.csv`.

**Three questions:**

1. Did the model get the two-score document right?
2. Where was the model more accurate than the regex, and where was the regex more
   accurate?
3. Neither the regex output nor the model output comes with any signal that a value is
   wrong. The schema is the check that finds wrong values. Which validation rules would
   have caught the model's mistakes?

Writing extraction code now takes almost no time. Checking whether the output is correct
is the real work, and that is why this lab is built around a schema and an answer key.

*Your answers.*

---
## Checkpoint: submit before leaving

1. Compare your extracted table to `inspections_a.csv` field by field. Which field was
   hardest to extract correctly, and how many of the 24 values did you get right for it?
2. What did the greedy `[\d.]+` pattern return, and why?
3. Which PDF contained two scores, which score is the correct one, and what rule catches
   the problem?
4. How many validation rules did your validator include, and which one caught something
   unexpected?
5. One sentence about P4: what is the least finished part of the project right now?

*Your answers.*

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Upload the notebook to **Week 12 In-Class Activity** on Moodle before the end of class on Friday. There is no take-home this week, so there is nothing to rename and no [AT HOME] submission.

---
## Reminders

**Lightning talks are Nov 13 and Nov 20.** Three minutes, presented from a working
notebook rather than a finished report. Spend about a third of the talk on the hardest
data problem you ran into and how you solved it; that part is what your classmates will
remember. If you have not chosen a talk day yet, sign up on Moodle this week.

**P4 is due Friday, Nov 20, 11:59 PM**: notebook, report, README, and the
AI use log. Before submitting, clone the repo into a fresh folder and run it
following your own README. A notebook that depends on a file outside the repo fails this
test, and it is much better to discover that yourself than to hear it from me.

**The Bench Check 2 window closes this week.** If you don't have a slot yet, take one
today or come to office hours.